# Data Cleaning & Quality Assessment

## Overview
1. Data Loading and Initial Inspection  
2. Duplicate Records (_id)  
3. Nested Field Normalization (spending_behavior)  
4. Data Quality Discovery (completeness, consistency, validity)  
5. Data Remediation (Cleaning Pipeline)  
6. Final Validation & Export  

## 1. Data Loading and Initial Inspection

This section loads the raw JSON file, flattens it into a tabular format, and performs basic inspection to understand schema and potential data quality issues.

**Cleaning Approach:**
We first stabilize the dataset structure by removing duplicate application records and normalizing nested fields.
After the dataset is fully tabular, we profile data quality issues (missing values, inconsistent formats, invalid values)
and apply a transparent, rule-based remediation pipeline.

In [1]:
# Imports and Data Loading

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

DATA_PATH = "../data/raw_credit_applications.json"  # adjust if needed

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.json_normalize(raw)

print("Loaded records:", len(raw))
print("DataFrame shape:", df.shape)
display(df.head(3))

Loaded records: 502
DataFrame shape: (502, 21)


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


**Initial data inspection** shows a total of 502 application records and 21 attributes.  
Several fields exhibit substantial missingness (e.g., processing_timestamp, loan_purpose, notes), while others are fully populated.  
Data types are heterogeneous, with multiple numeric attributes stored as object types, indicating potential consistency issues to be addressed in subsequent steps.

## 2. Duplicate Recoders (_id)

In [2]:
# Duplicate Records (_id)

if "_id" not in df.columns:
    raise KeyError("Expected column '_id' not found. Check your input JSON structure.")

dup_mask = df.duplicated(subset=["_id"], keep=False)
dup_rows = int(dup_mask.sum())

print("Rows with duplicate _id:", dup_rows)

if dup_rows > 0:
    # Show top repeated IDs
    display(
        df.loc[dup_mask, ["_id"]]
        .value_counts()
        .rename("count")
        .reset_index()
        .sort_values("count", ascending=False)
        .head(10)
    )

# Deduplicate: keep first occurrence
before = len(df)
df = df.drop_duplicates(subset=["_id"], keep="first").copy()
after = len(df)

print("Before dedup:", before)
print("After dedup :", after)
print("Dropped     :", before - after)

Rows with duplicate _id: 4


,_id,count
0,app_001,2
1,app_042,2


Before dedup: 502
After dedup : 500
Dropped     : 2


Duplicate application records were assessed using the application identifier.  
Hard duplicates were quantified and removed to ensure record uniqueness prior to further data quality remediation.

## 3. Nested Field Normalization (spending_behavior)


The original dataset contains a nested list field (`spending_behavior`) with category–amount pairs.
This structure is flattened into a single wide table by creating one column per spending category
(`spending_<category>`). After normalization, the dataset contains one row per application and no
nested fields remain.

In [3]:
# Flatten spending_behavior into spending_* columns (one flat table)

if "spending_behavior" not in df.columns:
    print("No 'spending_behavior' column found. Skipping.")
else:
    # Explode list -> rows (temporary)
    tmp = df[["_id", "spending_behavior"]].explode("spending_behavior")

    # Extract fields from each dict element
    tmp["category"] = tmp["spending_behavior"].apply(
        lambda x: x.get("category") if isinstance(x, dict) else np.nan
    )
    tmp["amount"] = tmp["spending_behavior"].apply(
        lambda x: x.get("amount") if isinstance(x, dict) else np.nan
    )
    tmp["amount"] = pd.to_numeric(tmp["amount"], errors="coerce")

    # Pivot to wide: one row per _id, one column per category
    wide = (
        tmp.pivot_table(index="_id", columns="category", values="amount", aggfunc="sum", fill_value=0)
        .reset_index()
    )

    # Make safe column names: spending_<category>
    def safe_spending_col(name):
        name = str(name).strip().replace(" ", "_")
        name = "".join(ch for ch in name if ch.isalnum() or ch == "_")
        return f"spending_{name}"

    wide = wide.rename(columns={c: safe_spending_col(c) for c in wide.columns if c != "_id"})

    # Drop nested column and merge spending columns
    df = df.drop(columns=["spending_behavior"]).merge(wide, on="_id", how="left")

    print("Flattened spending_behavior into columns.")
    print("New shape:", df.shape)

# sanity: confirm no list/dict columns remain
nested_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, (list, dict))).any()]
print("Nested columns remaining:", nested_cols)
display(df.head(3))

Flattened spending_behavior into columns.
New shape: (500, 35)
Nested columns remaining: []


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spending_Adult_Entertainment,spending_Alcohol,spending_Dining,spending_Education,spending_Entertainment,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0


## 4. Data Quality Discovery

After normalizing all nested fields, we assess data quality issues across four dimensions:

- **Completeness**: missing and hidden missing values  
- **Consistency**: formatting and data type inconsistencies  
- **Validity**: impossible or out-of-range values  
- **Uniqueness**: potential duplicate identifiers (e.g. SSN)

This step is purely diagnostic: no data is modified.

In [4]:
# Data Quality Discovery (NO MODIFICATIONS)

print("=== COMPLETENESS ===")

MISSING_TOKENS = {"", " ", "na", "n/a", "nan", "null", "none", "-", "--"}

standard_nulls = df.isna().sum()
hidden_nulls = df.map(
    lambda x: isinstance(x, str) and x.strip().lower() in MISSING_TOKENS
).sum()

missing_df = pd.DataFrame({
    "Standard Nulls": standard_nulls,
    "Hidden Nulls": hidden_nulls,
    "Total Missing": standard_nulls + hidden_nulls
})

display(
    missing_df[missing_df["Total Missing"] > 0]
    .sort_values("Total Missing", ascending=False)
)

print("\n=== CONSISTENCY (DATA TYPES & CATEGORIES) ===")
print("Object-type columns (potential inconsistencies):")
display(df.dtypes[df.dtypes == "object"])

for col in ["applicant_info.gender", "decision.rejection_reason"]:
    if col in df.columns:
        print(f"\nUnique values in {col}:")
        display(pd.Series(df[col].dropna().unique()))

print("\n=== VALIDITY (NUMERIC RANGES) ===")
numeric_cols = df.select_dtypes(include=[np.number]).columns
display(df[numeric_cols].describe().T[["min", "max", "mean"]])

print("\n=== UNIQUENESS (SSN) ===")
if "applicant_info.ssn" in df.columns:
    dup_ssn = df[df.duplicated(subset=["applicant_info.ssn"], keep=False)]
    print(
        f"Duplicate SSNs: {dup_ssn['applicant_info.ssn'].nunique()} "
        f"shared across {len(dup_ssn)} records"
    )

=== COMPLETENESS ===


,Standard Nulls,Hidden Nulls,Total Missing
notes,500,0,500
financials.annual_salary,495,0,495
loan_purpose,450,0,450
processing_timestamp,438,0,438
decision.rejection_reason,292,0,292
decision.interest_rate,208,0,208
decision.approved_amount,208,0,208
applicant_info.email,0,7,7
financials.annual_income,5,0,5
applicant_info.ssn,4,0,4



=== CONSISTENCY (DATA TYPES & CATEGORIES) ===
Object-type columns (potential inconsistencies):


_id                             object
processing_timestamp            object
applicant_info.full_name        object
applicant_info.email            object
applicant_info.ssn              object
applicant_info.ip_address       object
applicant_info.gender           object
applicant_info.date_of_birth    object
applicant_info.zip_code         object
financials.annual_income        object
decision.rejection_reason       object
loan_purpose                    object
notes                           object
dtype: object


Unique values in applicant_info.gender:


...


Unique values in decision.rejection_reason:


0           algorithm_risk_score
1    insufficient_credit_history
2                 high_dti_ratio
3                     low_income
dtype: object


=== VALIDITY (NUMERIC RANGES) ===


,min,max,mean
financials.credit_history_months,-10.00,133.00,50.444000
financials.debt_to_income,0.05,1.85,0.245520
financials.savings_balance,-5000.00,88078.00,29579.530000
decision.interest_rate,2.50,6.50,4.564726
decision.approved_amount,15000.00,80000.00,47845.890411
financials.annual_salary,45000.00,94000.00,69200.000000
spending_Adult_Entertainment,0.00,848.00,5.912000
spending_Alcohol,0.00,757.00,10.496000
spending_Dining,0.00,897.00,61.472000
spending_Education,0.00,889.00,63.956000



=== UNIQUENESS (SSN) ===
Duplicate SSNs: 2 shared across 8 records


- **High missingness** in optional/context fields: `notes` (500) and `processing_timestamp` (438).
- **Misaligned income fields**: `financials.annual_salary` is mostly missing (495), while `financials.annual_income` exists but is stored as `object` (needs numeric casting).
- **Categorical gaps**: `loan_purpose` has many missing values (450). `applicant_info.gender` contains inconsistent encodings (`Male`, `Female`, `M`, `F`, empty).
- **Decision-field missingness is conditional**: `decision.approved_amount` and `decision.interest_rate` are missing in 208 records (likely rejected cases).
- **Invalid values detected**: negative `financials.credit_history_months` and negative `financials.savings_balance` exist and must be handled.
- **Uniqueness issue**: 2 SSNs appear across 8 records (flag for review; do not automatically drop without a clear policy).

### Format Validity Checks (Regex-Based)

In addition to missing and range-based checks, we validate the **format correctness**
of selected identifier fields (email, SSN) using regular expressions.
This step identifies malformed values without modifying the dataset.

In [5]:
# Format Validity (Regex Checks) — Discovery Only

import re

print("=== FORMAT VALIDITY (REGEX CHECKS) ===")

# Email validation
EMAIL_REGEX = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

if "applicant_info.email" in df.columns:
    emails = df["applicant_info.email"].dropna().astype(str).str.strip()
    malformed_email_mask = ~emails.str.match(EMAIL_REGEX)

    print(f"Found {malformed_email_mask.sum()} malformed emails.")
    display(emails[malformed_email_mask].head(10))

# SSN validation (US format)
SSN_REGEX = r"^\d{3}-\d{2}-\d{4}$"

if "applicant_info.ssn" in df.columns:
    ssns = df["applicant_info.ssn"].dropna().astype(str).str.strip()
    malformed_ssn_mask = ~ssns.str.match(SSN_REGEX)

    print(f"Found {malformed_ssn_mask.sum()} malformed SSNs.")

=== FORMAT VALIDITY (REGEX CHECKS) ===
Found 11 malformed emails.


26                           
138    mike johnson@gmail.com
181     test.user.outlook.com
187                          
275                          
276          john.doe@invalid
297                          
298                          
368              sarah.smith@
447                          
Name: applicant_info.email, dtype: object

Found 0 malformed SSNs.


## 5. Data Remediation (Cleaning Pipeline)

We now apply a rule-based cleaning pipeline to address the issues identified in discovery:

- **Consistency**: standardize `gender`, cast data types, consolidate salary into income
- **Validity**: convert impossible negative values to missing (`NaN`)
- **Completeness**: fill missing categorical fields with `Unknown` where appropriate
- **Format validity**: invalidate malformed emails (set to `NaN`)
- **Uniqueness**: flag duplicate SSNs for review (do not drop automatically)

In [6]:
# Data Remediation (Cleaning Pipeline)

df_clean = df.copy()
remediation = {}

# Normalize hidden-missing tokens to NaN ---
MISSING_TOKENS = {"", " ", "na", "n/a", "nan", "null", "none", "-", "--"}

def normalize_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, float) and np.isnan(x):
        return np.nan
    if isinstance(x, str) and x.strip().lower() in MISSING_TOKENS:
        return np.nan
    return x

df_clean = df_clean.applymap(normalize_missing)
remediation["normalized_missing_tokens"] = "done"

# Fix annual_income type + merge annual_salary into annual_income ---
if "financials.annual_income" in df_clean.columns:
    df_clean["financials.annual_income"] = pd.to_numeric(df_clean["financials.annual_income"], errors="coerce")
else:
    df_clean["financials.annual_income"] = np.nan

if "financials.annual_salary" in df_clean.columns:
    missing_before = int(df_clean["financials.annual_income"].isna().sum())
    df_clean["financials.annual_income"] = df_clean["financials.annual_income"].fillna(df_clean["financials.annual_salary"])
    df_clean = df_clean.drop(columns=["financials.annual_salary"])
    remediation["merged_salary_into_income_missing_before"] = missing_before

# Cast numeric columns ---
num_cols = [
    "financials.credit_history_months",
    "financials.debt_to_income",
    "financials.savings_balance",
    "decision.approved_amount",
    "decision.interest_rate",
]
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Parse date_of_birth (mixed formats) ---
if "applicant_info.date_of_birth" in df_clean.columns:
    # dayfirst=True handles DD/MM/YYYY; ISO dates still parse correctly
    df_clean["applicant_info.date_of_birth"] = pd.to_datetime(
        df_clean["applicant_info.date_of_birth"], errors="coerce", dayfirst=True
    )
    remediation["parsed_date_of_birth"] = "done"

# Standardize gender + fill missing with Unknown ---
if "applicant_info.gender" in df_clean.columns:
    df_clean["applicant_info.gender"] = (
        df_clean["applicant_info.gender"]
        .astype("string")
        .str.strip()
        .replace({"M": "Male", "F": "Female", "m": "Male", "f": "Female"})
    )
    n_missing_gender = int(df_clean["applicant_info.gender"].isna().sum())
    df_clean["applicant_info.gender"] = df_clean["applicant_info.gender"].fillna("Unknown")
    remediation["filled_gender_unknown"] = n_missing_gender

# Fill loan_purpose missing with Unknown ---
if "loan_purpose" in df_clean.columns:
    n_missing_purpose = int(df_clean["loan_purpose"].isna().sum())
    df_clean["loan_purpose"] = df_clean["loan_purpose"].fillna("Unknown")
    remediation["filled_loan_purpose_unknown"] = n_missing_purpose

# Format validity: invalidate malformed emails (Option A) ---
import re
EMAIL_REGEX = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

if "applicant_info.email" in df_clean.columns:
    # strip whitespace first (doesn't "repair" beyond trimming)
    df_clean["applicant_info.email"] = df_clean["applicant_info.email"].astype("string").str.strip()

    email_series = df_clean["applicant_info.email"].dropna()
    malformed_mask = ~email_series.str.match(EMAIL_REGEX)

    remediation["malformed_emails_set_to_nan"] = int(malformed_mask.sum())

    # set malformed to NaN in df_clean
    bad_idx = email_series.index[malformed_mask]
    df_clean.loc[bad_idx, "applicant_info.email"] = np.nan

# Validity: negative values -> NaN ---
if "financials.savings_balance" in df_clean.columns:
    bad = df_clean["financials.savings_balance"] < 0
    remediation["savings_negative_to_nan"] = int(bad.sum())
    df_clean.loc[bad, "financials.savings_balance"] = np.nan

if "financials.credit_history_months" in df_clean.columns:
    bad = df_clean["financials.credit_history_months"] < 0
    remediation["credit_history_negative_to_nan"] = int(bad.sum())
    df_clean.loc[bad, "financials.credit_history_months"] = np.nan

# Logical decision consistency ---
if "decision.loan_approved" in df_clean.columns:
    approved = df_clean["decision.loan_approved"] == True
    rejected = df_clean["decision.loan_approved"] == False

    if "decision.rejection_reason" in df_clean.columns:
        # approved should not have rejection reason
        bad = approved & df_clean["decision.rejection_reason"].notna()
        remediation["cleared_rejection_reason_for_approved"] = int(bad.sum())
        df_clean.loc[bad, "decision.rejection_reason"] = np.nan

        # rejected with missing reason -> Unknown (optional, but clean for reporting)
        miss = rejected & df_clean["decision.rejection_reason"].isna()
        remediation["filled_rejection_reason_unknown_for_rejected"] = int(miss.sum())
        df_clean.loc[miss, "decision.rejection_reason"] = "Unknown"

    # rejected should not have approved fields
    for c in ["decision.approved_amount", "decision.interest_rate"]:
        if c in df_clean.columns:
            bad = rejected & df_clean[c].notna()
            remediation[f"cleared_{c}_for_rejected"] = int(bad.sum())
            df_clean.loc[bad, c] = np.nan

# Uniqueness: flag duplicate SSNs (do NOT drop) ---
if "applicant_info.ssn" in df_clean.columns:
    dup_ssn = df_clean["applicant_info.ssn"].notna() & df_clean.duplicated(subset=["applicant_info.ssn"], keep=False)
    df_clean["flag_duplicate_ssn"] = dup_ssn
    remediation["flag_duplicate_ssn_records"] = int(dup_ssn.sum())

# Report remediation summary ---
display(pd.DataFrame([remediation]).T.rename(columns={0: "count_or_note"}))
print("Shape after remediation:", df_clean.shape)
display(df_clean.head(3))

C:\Users\User\AppData\Local\Temp\ipykernel_20712\2992044576.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_clean = df_clean.applymap(normalize_missing)


,count_or_note
normalized_missing_tokens,done
merged_salary_into_income_missing_before,5
parsed_date_of_birth,done
filled_gender_unknown,2
filled_loan_purpose_unknown,450
malformed_emails_set_to_nan,4
savings_negative_to_nan,1
credit_history_negative_to_nan,2
cleared_rejection_reason_for_approved,0
filled_rejection_reason_unknown_for_rejected,0


Shape after remediation: (500, 35)


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,notes,spending_Adult_Entertainment,spending_Alcohol,spending_Dining,spending_Education,spending_Entertainment,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities,flag_duplicate_ssn
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-09-03,10036,73000.0,23.0,0.20,31212.0,False,algorithm_risk_score,Unknown,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0,False
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,NaT,10032,78000.0,51.0,0.18,17915.0,False,algorithm_risk_score,Unknown,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0,False
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,NaT,10075,61000.0,41.0,0.21,37909.0,True,NaN,vacation,3.7,59000.0,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0,False


## 6. Final Validation & Export

We perform a final sanity check on the cleaned dataset (shape, remaining missingness, basic validity signals),
then export the cleaned dataset for downstream analysis.

In [7]:
# Final Validation & Export

print("Final shape:", df_clean.shape)

# Quick missingness check (top 10)
missing_final = df_clean.isna().sum().sort_values(ascending=False)
missing_final = missing_final[missing_final > 0]
print("\nTop missing columns after remediation:")
display(missing_final.head(10))

# Confirm key fixes
print("\nChecks:")
print(" - annual_salary present?:", "financials.annual_salary" in df_clean.columns)
print(" - duplicate SSN flagged records:", int(df_clean.get("flag_duplicate_ssn", pd.Series(False)).sum()))

Final shape: (500, 35)

Top missing columns after remediation:


notes                               500
processing_timestamp                438
applicant_info.date_of_birth        357
decision.rejection_reason           292
decision.approved_amount            208
decision.interest_rate              208
applicant_info.email                 11
applicant_info.ssn                    4
applicant_info.ip_address             4
financials.credit_history_months      2
dtype: int64


Checks:
 - annual_salary present?: False
 - duplicate SSN flagged records: 4


In [10]:
# Export
OUT_CSV = "../data/processed/cleaned_credit_applications.csv"

df_clean.to_csv(OUT_CSV, index=False)


print("\nExported:")
print(" -", OUT_CSV)



Exported:
 - ../data/processed/cleaned_credit_applications.csv
